## Carga de datos

Los datos vienen presentados en dos archivos comprimidos, _protein.links_ y _protein.info_, el primero son los datos de interacciones entre proteínas (solo IDs), mientras que el segundo tiene la información de los aminoacidos que componen cada proteína.

In [1]:
from pyspark.sql import SparkSession

In [2]:
# Inicializar la sesión de Spark
spark = SparkSession.builder \
    .appName("AnalisisStringDB") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 09:37:12 WARN Utils: Your hostname, MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.93 instead (on interface en0)
26/04/13 09:37:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 09:37:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Cargar los datos
path = "../data/9606.protein.links.v12.0.txt.gz"
df = spark.read.option("header", "true") \
               .option("sep", " ") \
               .option("inferSchema", "true") \
               .csv(path)

df.show(5)

+--------------------+--------------------+--------------+
|            protein1|            protein2|combined_score|
+--------------------+--------------------+--------------+
|9606.ENSP00000000233|9606.ENSP00000356607|           173|
|9606.ENSP00000000233|9606.ENSP00000427567|           154|
|9606.ENSP00000000233|9606.ENSP00000253413|           151|
|9606.ENSP00000000233|9606.ENSP00000493357|           471|
|9606.ENSP00000000233|9606.ENSP00000324127|           201|
+--------------------+--------------------+--------------+
only showing top 5 rows


## Operaciones con PySpark

In [5]:
# Filtrar registros con puntuación combinada alta
df_high_confidence = df.filter(df.combined_score >= 700)

print(f"Registros originales: {df.count()}")
print(f"Registros alta confianza: {df_high_confidence.count()}")

Registros originales: 13715404


Registros alta confianza: 473860


In [6]:
# Estadísticas descriptivas de la columna combined_score
df.select("combined_score").describe().show()

+-------+------------------+
|summary|    combined_score|
+-------+------------------+
|  count|          13715404|
|   mean| 268.6240444685406|
| stddev|156.98280103935286|
|    min|               150|
|    max|               999|
+-------+------------------+



### Operación aritmética
Se quiere normalizar el score o calcular un "score de confianza relativa" respecto al máximo teórico (1000).

In [7]:
from pyspark.sql.functions import col

In [9]:
df_processed = df.withColumn("normalized_score", col("combined_score") / 1000) \
                 .withColumn("confidence_gap", 1.0 - (col("combined_score") / 1000))

df_processed.select("protein1", "protein2", "combined_score", "normalized_score").show(5)

+--------------------+--------------------+--------------+----------------+
|            protein1|            protein2|combined_score|normalized_score|
+--------------------+--------------------+--------------+----------------+
|9606.ENSP00000000233|9606.ENSP00000356607|           173|           0.173|
|9606.ENSP00000000233|9606.ENSP00000427567|           154|           0.154|
|9606.ENSP00000000233|9606.ENSP00000253413|           151|           0.151|
|9606.ENSP00000000233|9606.ENSP00000493357|           471|           0.471|
|9606.ENSP00000000233|9606.ENSP00000324127|           201|           0.201|
+--------------------+--------------------+--------------+----------------+
only showing top 5 rows


Operación **JOIN** del archivo de información

In [11]:
# Cargar el archivo de información
info_path = "../data/9606.protein.info.v12.0.txt.gz"

# En este archivo, STRING suele usar tabuladores (\t)
df_info = spark.read.option("header", "true") \
                    .option("sep", "\t") \
                    .option("inferSchema", "true") \
                    .csv(info_path)

# Renombrar la columna de ID para facilitar el join y evitar ambigüedad
# La columna original suele llamarse #string_protein_id
df_info = df_info.withColumnRenamed("#string_protein_id", "protein_id")

df_info.show(5)

+--------------------+--------------+------------+--------------------+
|          protein_id|preferred_name|protein_size|          annotation|
+--------------------+--------------+------------+--------------------+
|9606.ENSP00000000233|          ARF5|         180|ADP-ribosylation ...|
|9606.ENSP00000000412|          M6PR|         277|Cation-dependent ...|
|9606.ENSP00000001008|         FKBP4|         459|Peptidyl-prolyl c...|
|9606.ENSP00000001146|       CYP26B1|         512|Cytochrome P450 2...|
|9606.ENSP00000002125|       NDUFAF7|         441|Protein arginine ...|
+--------------------+--------------+------------+--------------------+
only showing top 5 rows


In [12]:
# Realizar el Join
# Queremos los datos de df_processed + el nombre de la proteína que coincide con protein1
df_final = df_processed.join(df_info, df_processed.protein1 == df_info.protein_id, "inner") \
                       .select("preferred_name", "protein2", "combined_score", "normalized_score") \
                       .withColumnRenamed("preferred_name", "name_p1")

df_final.show(5)

+-------+--------------------+--------------+----------------+
|name_p1|            protein2|combined_score|normalized_score|
+-------+--------------------+--------------+----------------+
|   ARF5|9606.ENSP00000356607|           173|           0.173|
|   ARF5|9606.ENSP00000427567|           154|           0.154|
|   ARF5|9606.ENSP00000253413|           151|           0.151|
|   ARF5|9606.ENSP00000493357|           471|           0.471|
|   ARF5|9606.ENSP00000324127|           201|           0.201|
+-------+--------------------+--------------+----------------+
only showing top 5 rows


In [13]:
spark.stop()